This first code will run a 'grid search' on the tiny sample of 79 MOA-printed chemicals as matched from the list of 100 that Grace gave me. It will not be a true grid search, as we won't use any cross-validation due to tiny sample size, but I will iterate through various parameters. 

In [7]:
#Read in the ld50 data
import pandas as pd

data = pd.read_csv('../src/data/moa_sample_pre-grid-search.csv').set_index('Unnamed: 0', drop = True)
data.head()

,LD50_LM,tp_fp0,tp_fp1,tp_fp2,tp_fp3,tp_fp4,tp_fp5,tp_fp6,tp_fp7,tp_fp8,...,sc_Triazolinones,sc_Triazolocarboxamide,sc_Triazolopyrimidine_-_Type_1,sc_Triazolopyrimidine_-_Type_2,sc_Trifluoromethanesulfonanilides,sc_Triketones,sc_Triketones_(procide),sc_Uracils,sc_Ureas,sc_alpha-Chloroacetamides
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
DTXSID6024177,1.183447,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DTXSID7022253,-0.565330,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DTXSID7034961,-0.052205,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
DTXSID4032619,-0.935266,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DTXSID7021948,0.707871,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
#Generate the parameter list for hybrid weights
from numpy import arange
fp_list = ["A", "B", "C", "D", "E"]
increment = 10
weight_list = list(arange(0,100+increment, increment))
params = []
for a in weight_list:
    for b in weight_list:
        if a + b > 100:
            password
        elif a + b == 100:
            params.append([a,b,0,0,0])
        else:
            for c in weight_list:
                if a+b+c > 100:
                    password
                elif a+b+c == 1000:
                    params.append([a,b,c,0,0])
                else:
                    for d in weight_list:
                        if a+b+c+d > 100:
                            password
                        elif a+b+c+d == 100:
                            params.append([a,b,c,d,0])
                        else:
                            for e in weight_list:
                                if a+b+c+d+e == 100:
                                    params.append([a,b,c,d,e])
                                else:
                                    password
print(len(params))

1001


In [33]:
from sklearn.model_selection import ParameterGrid

#Parameters for Grid Search
hybrid_weights = params
neighborhood_size = list(range(4,14))

params = {'hybrid_weights':hybrid_weights, 'n_neighbors': neighborhood_size}

#Parameters for set-up and record-keeping
slices = [slice(0,729),slice(729,2777), slice(2777,4825), slice(4825, 5727), slice(5727, None)]

In [35]:
#Search through all the possible print, neighborhood combinations
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from genra.rax.skl.hybrid import GenRAPredValueHybrid
import numpy as np
from numpy import sqrt

params_list = ParameterGrid(params)
tester = GenRAPredValueHybrid(n_neighbors =8, slices = slices)



results = {'print':[], 'n_neighbors':[], 'rmse':[], 'r2':[], 'nulls':[]}
for i in list(params_list):

    results['n_neighbors'].append(i['n_neighbors'])
    results['print'].append(i['hybrid_weights'])

    for key, value in i.items():
        setattr(tester, key, value)


    x_test = data.iloc[:,1:]
    y_test = data.iloc[:,0]

    y_preds = []

    for x, prints in x_test.iterrows():
        tester.fit(data.loc[data.index != x].iloc[:, 1:], data.loc[data.index != x].iloc[:,0])
        y_preds.append(tester.predict(prints.to_numpy().reshape(1,-1)))
    scorable_preds = []
    scorable_tests = []
    null_counter = 0
    for i in range(len(y_preds)):
        if np.isnan(y_preds[i]):
            null_counter += 1
            print(i)
        else:
            scorable_preds.append(y_preds[i])
            scorable_tests.append(y_test[i])
    r2 = r2_score(scorable_tests, scorable_preds)
    rmse = float(sqrt((sum([(scorable_preds[i]-scorable_tests[i])**2 for i in range(0,len(scorable_preds))])/len(scorable_preds))))
    results['r2'].append(r2)
    results['rmse'].append(rmse)
    results['nulls'].append(null_counter)

results_df = pd.DataFrame(results)
results_df.to_csv(f'../references/moa/sample_tests.csv')

c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid

0
1
2
4
6
7
10
13
14
15
16
17
18
20
21
22
23
24
25
27
28
29
30
31
32
33
34
35
36
37
38
40
41
42
44
45
46
47
49
50
51
52
53
54
55
57
58
59
61
62
63
64
68
71
72
73
74
75
76
78


c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid

0
1
4
6
7
10
13
14
17
18
21
22
23
24
25
27
28
29
30
31
32
33
34
35
36
37
38
40
41
42
44
45
46
47
50
51
52
53
54
55
57
58
59
61
62
63
64
68
71
72
73
74
75
76
78


C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rmse = float(sqrt((sum([(scorable_preds[i]-scorable_tests[i])**2 for i in range(0,len(scorable_preds))])/len(scorable_preds))))
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental Protection Agency (EPA)\Profile\Documents\Grid Search Optimization\genra-hybrid\venv\Lib\site-packages\genra\rax\skl\reg.py:120: RuntimeWarning: invalid value encountered in divide
  neigh_dist_n = neigh_dist / neigh_dist.max()
c:\Users\aleary\OneDrive - Environmental P

...

C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  scorable_tests.append(y_test[i])
C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rmse = float(sqrt((sum([(scorable_preds[i]-scorable_tests[i])**2 for i in range(0,len(scorable_preds))])/len(scorable_preds))))
C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behav

26


C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  scorable_tests.append(y_test[i])
C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rmse = float(sqrt((sum([(scorable_preds[i]-scorable_tests[i])**2 for i in range(0,len(scorable_preds))])/len(scorable_preds))))
C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behav

26


C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  scorable_tests.append(y_test[i])
C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rmse = float(sqrt((sum([(scorable_preds[i]-scorable_tests[i])**2 for i in range(0,len(scorable_preds))])/len(scorable_preds))))
C:\Users\aleary\AppData\Local\Temp\ipykernel_22132\1884486029.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behav

In [4]:
#Read in the results from the previous test if kernel has restarted since
import pandas as pd
results_df = pd.read_csv(f'../references/moa/sample_tests.csv').set_index("Unnamed: 0")

In [5]:
#View the best results from the test
results_df.sort_values('r2', ascending=False).head(15)

,print,n_neighbors,rmse,r2,nulls
Unnamed: 0,,,,,
662,"[0, 10, 0, 0, 90]",6,0.779278,0.389899,0
2823,"[0, 90, 0, 0, 10]",7,0.780396,0.388147,0
5022,"[10, 80, 0, 0, 10]",6,0.782753,0.384446,0
5023,"[10, 80, 0, 0, 10]",7,0.783441,0.383364,0
1212,"[0, 20, 0, 0, 80]",6,0.783538,0.383212,0
2853,"[0, 100, 0, 0, 0]",7,0.784306,0.382002,0
2793,"[0, 80, 10, 0, 10]",7,0.784687,0.381401,0
663,"[0, 10, 0, 0, 90]",7,0.786025,0.379290,0
6673,"[20, 70, 0, 0, 10]",7,0.786175,0.379053,0
